# Import related pacakages

In [1]:
import sys
sys.path.append(r'./ETGEM/etgems_web/')
from ETGEM.etgems_web.script.io.json import read_json_model
from ETGEM.etgems_web.script.analysis.flux_balance_analysis import calculate_ETM
import pyomo.environ as pyo
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

MODEL_PATH = "./model/ETiZM564_MM_optimized_23BDO.json"
output_file = "FVSEOF_results_v2_564.xlsx"
BIOMASS_ID = "ZM_BIOMASS"
TARGET_PRODUCT = "EX_23BDO_e"
SUBSTRATE = "EX_glc_D_e_reverse"
SUBSTRATE_VALUE = 100
N_SCAN_STEPS = 15

MAX_BDO_INITIAL = 1000  # 初始 max_bdo 估值（无氧计算后覆盖）
O2_FEEDBACK_BETA = 1
O2_LEVEL =6
TARGET_BDO_RXN = "rxn02112_reverse"  # 23BDO 生产反应（被反馈约束限制）
FLUX_THRESHOLD = 0.01


def setup_model(o2_level=O2_LEVEL):
    model = read_json_model(MODEL_PATH)
    model.add_medium("EX_glc_D_e", 0)
    model.get_reaction_by_id('EX_glc_D_e').bounds = (0, 0)
    model.add_medium(SUBSTRATE, SUBSTRATE_VALUE)

    for rxn in model.reactions:
        if "ZMO1360" in model.get_reaction_by_id(rxn).gene_reaction_rule:
            model.get_reaction_by_id(rxn).bounds = (0, 0)

    model.get_reaction_by_id("EX_lac_D_e").bounds = (0, 0)

    for o2rxn in ["rxn00262",'rxn00208']:
        model.get_reaction_by_id(o2rxn).bounds = (0, 0)
    model.add_medium("EX_o2_e_reverse", o2_level)
    model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
    model.get_reaction_by_id("rxn37453").kcat = 15
    model.get_reaction_by_id("rxn37453").kcat_MW = 15

    return model


def make_o2_feedback_constraint(alpha):
    def rule(m):
        return m.reaction[TARGET_PRODUCT] <= alpha
    return rule


def get_all_flux_values(result, reaction_list):
    fluxes = {}
    for rxn_id in reaction_list:
        try:
            val = result.reaction[rxn_id].value
            fluxes[rxn_id] = val if val is not None else 0.0
        except:
            fluxes[rxn_id] = 0.0
    return fluxes

In [2]:



def run_fvseof():
    print("=" * 70)
    print("FVSEOF 分析 — 寻找提高 23BDO 产量的靶标基因")
    print(f"  (含氧气反馈约束: alpha = max_bdo*1.3 - {O2_FEEDBACK_BETA} * O2)")
    print(f"  固定 O2 = {O2_LEVEL}")
    print("=" * 70)

    model = setup_model()
    reaction_list = sorted(list(model.reactions.keys()))

    # ====== Phase 0: 无氧计算 max_bdo ======
    print(f"\n[Phase 0] 无氧 (O2=0) 条件下计算 max_bdo (2,3-BDO的最大通量)...")
    anaerobic_model = setup_model(o2_level=0)
    anaerobic_model.obj_name = TARGET_BDO_RXN
    result_anaerobic = calculate_ETM(anaerobic_model, "maximize", 0.0)
    max_bdo = result_anaerobic.reaction[TARGET_BDO_RXN].value
    if max_bdo is None:
        max_bdo = 0.0
    print(f"  无氧 max_bdo = {max_bdo:.4f}")

    wt_alpha = max_bdo * 1.3 - O2_LEVEL * O2_FEEDBACK_BETA
    print(f"  反馈 alpha (O2={O2_LEVEL}) = max_bdo*1.3 - {O2_FEEDBACK_BETA}*O2 = {wt_alpha:.4f}")

    # print("\n[Phase 1] 扫描不同氧气水平下的生长与BDO...")
    # o2_levels = list(range(0, 21, 2))
    # o2_scan_results = []

    # for o2 in o2_levels:
    #     scan_m = setup_model(o2_level=o2)
    #     scan_m.obj_name = BIOMASS_ID
    #     alpha = max_bdo * 1.3 - o2 * O2_FEEDBACK_BETA

    #     custom = {'bdo_limit': make_o2_feedback_constraint(alpha)}
    #     result = calculate_ETM(scan_m, "maximize", 0.0, custom_constraints=custom)

    #     bio = result.reaction[BIOMASS_ID].value
    #     bdo = result.reaction[TARGET_PRODUCT].value
    #     o2_scan_results.append({'o2': o2, 'alpha': alpha, 'bio': bio, 'bdo': bdo})
    #     print(f"  O2={o2:2d} | alpha={alpha:.4f} | 生长={bio:.5f} | BDO={bdo:.4f}")

    # o2_df = pd.DataFrame(o2_scan_results)

    # ====== Phase 1: 在固定O2下计算WT基准 ======
    print(f"\n[Phase 1] 在 O2={O2_LEVEL} 下计算WT基准 (含反馈约束)...")
    wt_model = setup_model(o2_level=O2_LEVEL)
    wt_model.obj_name = BIOMASS_ID
    result_wt = calculate_ETM(wt_model, "maximize", 0.0,
                              custom_constraints={'bdo_limit': make_o2_feedback_constraint(wt_alpha)})
    WTbiomass = result_wt.reaction[BIOMASS_ID].value
    wt_fluxes = get_all_flux_values(result_wt, reaction_list)
    wt_fluxes_series = pd.Series(wt_fluxes)
    print(f"  WT 生长速率 = {WTbiomass:.6f}")
    print(f"  WT 23BDO = {wt_fluxes[TARGET_PRODUCT]:.4f}")

    # ====== Phase 3: 计算理论最大23BDO ======
    wt_bdo_model = setup_model(o2_level=O2_LEVEL)
    wt_bdo_model.obj_name = TARGET_PRODUCT

    def bio_min_constraint(m):
        return m.reaction[BIOMASS_ID] >= 0.001

    result_max_bdo = calculate_ETM(wt_bdo_model, "maximize", 0.0,
                                   custom_constraints={
                                       'bdo_limit': make_o2_feedback_constraint(wt_alpha),
                                       'bio_min': bio_min_constraint,
                                   })
    max_23bdo = result_max_bdo.reaction[TARGET_PRODUCT].value
    print(f"  理论最大23BDO = {max_23bdo:.4f}")

    # ====== Phase 2: FVSEOF扫描 ======
    print(f"\n[Phase 2] FVSEOF 扫描 ({N_SCAN_STEPS} 步)...")
    print(f"  策略: 在 O2={O2_LEVEL}, alpha={wt_alpha:.4f} 下，强制 BDO 从WT水平到理论最大值")

    wt_bdo = wt_fluxes[TARGET_PRODUCT]
    bdo_levels = np.linspace(wt_bdo * 1.0, max_23bdo * 0.98, N_SCAN_STEPS)
    print(f"  扫描范围: [{bdo_levels[0]:.2f}, {bdo_levels[-1]:.2f}]")

    scan_fluxes = {}
    for idx, bdo_level in enumerate(bdo_levels):
        scan_model = setup_model(o2_level=O2_LEVEL)
        scan_model.obj_name = BIOMASS_ID

        def make_bdo_constraint(target):
            def rule(m):
                return m.reaction[TARGET_PRODUCT] >= target
            return rule

        custom_scan = {
            'bdo_limit': make_o2_feedback_constraint(wt_alpha),
            'bdo_min': make_bdo_constraint(bdo_level),
        }
        result_scan = calculate_ETM(scan_model, "maximize", 0.0, custom_constraints=custom_scan)

        scan_fluxes[f"step_{idx}"] = get_all_flux_values(result_scan, reaction_list)
        bio_actual = result_scan.reaction[BIOMASS_ID].value
        bdo_actual = result_scan.reaction[TARGET_PRODUCT].value
        print(f"  步{idx+1:2d}: BDO≥{bdo_level:6.2f} | 实际BDO={bdo_actual:6.2f} | 生长={bio_actual:.5f}")

    # ====== Phase 3: 分析通量趋势 ======
    print("\n[Phase 3] 分析通量变化趋势...")
    scan_df = pd.DataFrame(scan_fluxes, index=reaction_list)
    scan_df.index.name = 'reaction_id'

    # 过滤掉WT通量接近0的反应（避免除零伪影）
    valid_mask = wt_fluxes_series.abs() > FLUX_THRESHOLD
    valid_reactions = valid_mask[valid_mask].index
    print(f"  WT通量 > {FLUX_THRESHOLD} 的反应数: {len(valid_reactions)} (过滤前 {len(reaction_list)})")

    scan_df_valid = scan_df.loc[scan_df.index.isin(valid_reactions)].copy()

    fc_columns = []
    for sidx in range(N_SCAN_STEPS):
        col = f"FC_step{sidx}"
        fc_values = scan_df_valid[f"step_{sidx}"] / wt_fluxes_series[scan_df_valid.index]
        fc_values = fc_values.replace([np.inf, -np.inf], np.nan).fillna(1.0)
        scan_df_valid[col] = fc_values
        fc_columns.append(col)

    # 获取GPR
    gpr_map = {}
    for rid in reaction_list:
        try:
            gpr_map[rid] = model.reactions[rid].gene_reaction_rule
        except:
            gpr_map[rid] = ''

    scan_df_valid['GPR'] = scan_df_valid.index.map(gpr_map)
    scan_df_gpr = scan_df_valid.dropna(subset=['GPR'])
    scan_df_gpr = scan_df_gpr[scan_df_gpr['GPR'] != '']
    scan_df_gpr = scan_df_gpr[scan_df_gpr['GPR'] != 's0001']

    # 用线性回归斜率判断趋势
    x = np.arange(N_SCAN_STEPS)
    slopes = []
    r_squared = []
    for rid in scan_df_gpr.index:
        y = scan_df_gpr.loc[rid, fc_columns].values.astype(float)
        slope = np.polyfit(x, y, 1)[0]
        slopes.append(slope)
        ss_res = np.sum((y - (slope * x + np.polyfit(x, y, 1)[1])) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        r_squared.append(r2)

    scan_df_gpr['FC_slope'] = slopes
    scan_df_gpr['R_squared'] = r_squared
    scan_df_gpr['FC_mean_tail'] = scan_df_gpr[fc_columns[-3:]].mean(axis=1)
    scan_df_gpr['FC_mean_head'] = scan_df_gpr[fc_columns[:3]].mean(axis=1)
    scan_df_gpr['FC_range'] = scan_df_gpr[fc_columns].max(axis=1) - scan_df_gpr[fc_columns].min(axis=1)

    # 分类：要求斜率显著且R²较高（趋势一致）
    slope_threshold = 0.005
    r2_threshold = 0.5

    up_reactions = scan_df_gpr[
        (scan_df_gpr['FC_slope'] > slope_threshold) &
        (scan_df_gpr['R_squared'] > r2_threshold)
    ].copy()
    down_reactions = scan_df_gpr[
        (scan_df_gpr['FC_slope'] < -slope_threshold) &
        (scan_df_gpr['R_squared'] > r2_threshold)
    ].copy()

    up_reactions['regulation'] = 'up'
    down_reactions['regulation'] = 'down'

    print(f"  有效反应数 (有GPR): {len(scan_df_gpr)}")
    print(f"  上调靶标反应数 (斜率>{slope_threshold}, R²>{r2_threshold}): {len(up_reactions)}")
    print(f"  下调靶标反应数 (斜率<-{slope_threshold}, R²>{r2_threshold}): {len(down_reactions)}")

    # ====== Phase 5: 基因级别分析 ======
    print("\n[Phase 5] 基因级别靶标识别...")

    def extract_genes_from_gpr(gpr_string):
        if not gpr_string or gpr_string == '':
            return []
        gpr_clean = gpr_string.replace(' and ', ' or ')
        genes_raw = gpr_clean.split(' or ')
        genes = []
        for g in genes_raw:
            g = g.strip().replace('(', '').replace(')', '')
            if g and g != '':
                genes.append(g)
        return list(set(genes))

    all_results = pd.concat([up_reactions, down_reactions])

    gene_rows = []
    for rid, row in all_results.iterrows():
        genes = extract_genes_from_gpr(row['GPR'])
        for gene in genes:
            gene_rows.append({
                'gene_id': gene,
                'reaction_id': rid,
                'regulation': row['regulation'],
                'GPR': row['GPR'],
                'FC_slope': row['FC_slope'],
                'R_squared': row['R_squared'],
                'FC_mean_tail': row['FC_mean_tail'],
                'FC_range': row['FC_range'],
            })

    gene_df = pd.DataFrame(gene_rows) if gene_rows else pd.DataFrame()

    if not gene_df.empty:
        gene_summary = gene_df.groupby('gene_id').agg(
            num_reactions=('reaction_id', 'nunique'),
            reactions=('reaction_id', lambda x: ','.join(x.unique())),
            regulation_types=('regulation', lambda x: list(x.unique())),
            FC_slope_mean=('FC_slope', 'mean'),
            R_squared_mean=('R_squared', 'mean'),
            FC_mean_tail=('FC_mean_tail', 'mean'),
            FC_range=('FC_range', 'max'),
        ).reset_index()

        gene_summary['primary_regulation'] = gene_summary['regulation_types'].apply(
            lambda x: 'up' if 'up' in x and 'down' not in x
            else 'down' if 'down' in x and 'up' not in x
            else 'mixed'
        )

        target_up = gene_summary[gene_summary['primary_regulation'] == 'up'].sort_values(
            'FC_slope_mean', ascending=False)
        target_down = gene_summary[gene_summary['primary_regulation'] == 'down'].sort_values(
            'FC_slope_mean', ascending=True)

        target_up['priority_score'] = target_up['FC_slope_mean'] * target_up['num_reactions'] * target_up['R_squared_mean']
        target_up = target_up.sort_values('priority_score', ascending=False)

        target_down['priority_score'] = abs(target_down['FC_slope_mean']) * target_down['num_reactions'] * target_down['R_squared_mean']
        target_down = target_down.sort_values('priority_score', ascending=False)
    else:
        target_up = pd.DataFrame()
        target_down = pd.DataFrame()

    # ========== 结果输出 ==========
    print("\n" + "=" * 70)
    print("FVSEOF 结果汇总 (含氧气反馈约束)")
    print("=" * 70)

    # print(f"\n--- O2-BDO 关系 (Phase 1) ---")
    # print(o2_df.to_string(index=False))

    print(f"\n--- 上调靶标基因 (建议过表达) Top25 ---")
    if len(target_up) > 0:
        for _, row in target_up.head(25).iterrows():
            print(f"  {row['gene_id']:15s} | {int(row['num_reactions']):3d}个反应 "
                  f"| 斜率={row['FC_slope_mean']:.5f} "
                  f"| R²={row['R_squared_mean']:.3f} "
                  f"| FC范围={row['FC_range']:.3f} "
                  f"| 优先级={row['priority_score']:.4f}")
    else:
        print("  (无)")

    print(f"\n--- 下调靶标基因 (建议敲除/抑制) Top25 ---")
    if len(target_down) > 0:
        for _, row in target_down.head(25).iterrows():
            print(f"  {row['gene_id']:15s} | {int(row['num_reactions']):3d}个反应 "
                  f"| 斜率={row['FC_slope_mean']:.5f} "
                  f"| R²={row['R_squared_mean']:.3f} "
                  f"| FC范围={row['FC_range']:.3f} "
                  f"| 优先级={row['priority_score']:.4f}")
    else:
        print("  (无)")

    # 保存到文件
    with pd.ExcelWriter(output_file) as writer:
        # o2_df.to_excel(writer, sheet_name='O2扫描结果', index=False)
        if len(target_up) > 0:
            target_up.to_excel(writer, sheet_name='上调靶标基因', index=False)
        if len(target_down) > 0:
            target_down.to_excel(writer, sheet_name='下调靶标基因', index=False)
        if not gene_df.empty:
            gene_df.to_excel(writer, sheet_name='基因级明细', index=False)
        up_reactions.to_excel(writer, sheet_name='上调靶标反应', index=True)
        down_reactions.to_excel(writer, sheet_name='下调靶标反应', index=True)
        scan_df_gpr.to_excel(writer, sheet_name='全部FC数据', index=True)
    print(f"\n结果已保存至: {output_file}")

    return {
        # 'o2_scan': o2_df,
        'target_genes_up': target_up,
        'target_genes_down': target_down,
        'up_reactions': up_reactions,
        'down_reactions': down_reactions,
        'scan_fluxes': scan_fluxes,
        'wt_fluxes': wt_fluxes,
    }


if __name__ == "__main__":
    results = run_fvseof()

FVSEOF 分析 — 寻找提高 23BDO 产量的靶标基因
  (含氧气反馈约束: alpha = max_bdo*1.3 - 1 * O2)
  固定 O2 = 6

[Phase 0] 无氧 (O2=0) 条件下计算 max_bdo (2,3-BDO的最大通量)...
  无氧 max_bdo = 27.9787
  反馈 alpha (O2=6) = max_bdo*1.3 - 1*O2 = 30.3723

[Phase 1] 在 O2=6 下计算WT基准 (含反馈约束)...
  WT 生长速率 = 0.059187
  WT 23BDO = 24.6009
  理论最大23BDO = 27.5371

[Phase 2] FVSEOF 扫描 (15 步)...
  策略: 在 O2=6, alpha=30.3723 下，强制 BDO 从WT水平到理论最大值
  扫描范围: [24.60, 26.99]
  步 1: BDO≥ 24.60 | 实际BDO= 24.60 | 生长=0.05919
  步 2: BDO≥ 24.77 | 实际BDO= 24.77 | 生长=0.05658
  步 3: BDO≥ 24.94 | 实际BDO= 24.94 | 生长=0.05393
  步 4: BDO≥ 25.11 | 实际BDO= 25.11 | 生长=0.05128
  步 5: BDO≥ 25.28 | 实际BDO= 25.28 | 生长=0.04863
  步 6: BDO≥ 25.45 | 实际BDO= 25.45 | 生长=0.04598
  步 7: BDO≥ 25.62 | 实际BDO= 25.62 | 生长=0.04333
  步 8: BDO≥ 25.79 | 实际BDO= 25.79 | 生长=0.04068
  步 9: BDO≥ 25.96 | 实际BDO= 25.96 | 生长=0.03694
  步10: BDO≥ 26.13 | 实际BDO= 26.13 | 生长=0.03305
  步11: BDO≥ 26.30 | 实际BDO= 26.30 | 生长=0.02915
  步12: BDO≥ 26.48 | 实际BDO= 26.48 | 生长=0.02525
  步13: BDO≥ 26.65 | 实际BDO= 26.65 |